In [1]:
pip install kafka-python

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import json
import time
import random
from kafka import KafkaProducer

# ==========================================
# 1. CONFIGURACIÓN DEL SIMULADOR
# ==========================================
KAFKA_BROKER = 'kafka:9093'  # El puerto interno de tu red Docker
KAFKA_TOPIC = 'transactions'

# CSV de Kaggle
CSV_PATH = '/home/jovyan/work/data/creditcard.csv' 

# ==========================================
# 2. INICIALIZAR EL PRODUCTOR KAFKA
# ==========================================
print(f"Conectando al broker Kafka en {KAFKA_BROKER}...")
producer = KafkaProducer(
    bootstrap_servers=[KAFKA_BROKER],
    # Serializamos automáticamente el diccionario de Python a JSON binario
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)
print("¡Conexión establecida!")

# ==========================================
# 3. CARGA Y PREPARACIÓN DE DATOS
# ==========================================
print(f"Cargando dataset desde {CSV_PATH}...")
df = pd.read_csv(CSV_PATH)

# Separamos los datos para poder manipular la probabilidad de inyección
df_fraud = df[df['Class'] == 1]
df_normal = df[df['Class'] == 0]

print(f"Dataset cargado. Fraudes totales: {len(df_fraud)} | Transacciones normales: {len(df_normal)}")
print("Iniciando inyección de datos en streaming (Presiona 'Stop' o Ctrl+C para detener)...")
print("-" * 50)

# ==========================================
# 4. BUCLE DE SIMULACIÓN (STREAMING)
# ==========================================
try:
    while True:
        if random.random() < 0.2 and not df_fraud.empty:
            # Escogemos una fila aleatoria de los fraudes
            row = df_fraud.sample(1).iloc[0]
            tx_type = "FRAUDE"
        else:
            # Escogemos una fila aleatoria de las normales
            row = df_normal.sample(1).iloc[0]
            tx_type = "NORMAL"

        # Convertimos la fila a diccionario y eliminamos la columna 'Class'
        payload = row.drop('Class').to_dict()

        # Enviamos el JSON por Kafka
        producer.send(KAFKA_TOPIC, value=payload)
        
        # Forzamos el envío inmediato
        producer.flush()

        # Imprimimos por pantalla para que veas qué está saliendo
        print(f"Enviado {tx_type} | Time: {payload.get('Time', 0):.0f} | Amount: ${payload.get('Amount', 0):.2f}")
        
        # Esperamos 1 segundo entre transacciones para simular tráfico real
        time.sleep(0.2)

except KeyboardInterrupt:
    print("\nSimulación detenida por el usuario.")
except Exception as e:
    print(f"\nError en el simulador: {e}")
finally:
    producer.close()
    print("Conexión con Kafka cerrada.")

Conectando al broker Kafka en kafka:9093...
¡Conexión establecida!
Cargando dataset desde /home/jovyan/work/data/creditcard.csv...
Dataset cargado. Fraudes totales: 492 | Transacciones normales: 284315
Iniciando inyección de datos en streaming (Presiona 'Stop' o Ctrl+C para detener)...
--------------------------------------------------
Enviado FRAUDE | Time: 94625 | Amount: $33.76
Enviado NORMAL | Time: 137705 | Amount: $20.42
Enviado NORMAL | Time: 53062 | Amount: $0.89
Enviado NORMAL | Time: 117110 | Amount: $15.03
Enviado FRAUDE | Time: 48380 | Amount: $208.58
Enviado NORMAL | Time: 147334 | Amount: $108.11
Enviado FRAUDE | Time: 68207 | Amount: $1.00
Enviado NORMAL | Time: 127860 | Amount: $8.00
Enviado NORMAL | Time: 123597 | Amount: $86.77
Enviado NORMAL | Time: 75861 | Amount: $2.58
Enviado NORMAL | Time: 93629 | Amount: $1.98
Enviado FRAUDE | Time: 93879 | Amount: $30.31
Enviado NORMAL | Time: 160875 | Amount: $5.00
Enviado NORMAL | Time: 41951 | Amount: $12.88
Enviado NORMAL |

# Producer 

## 1. Instalar el cliente de Kafka dentro de Jupyter
!pip install kafka-python-ng pandas

import time
import json
import pandas as pd
from kafka import KafkaProducer

# 2. Configurar la conexión. 
# ATENCIÓN: Como estamos dentro de la red de Docker, usamos el listener INTERNAL (puerto 9093)
BOOTSTRAP_SERVERS = 'kafka:9093'
TOPIC_NAME = 'transactions'
CSV_PATH = '../data/creditcard.csv'

producer = KafkaProducer(
    bootstrap_servers=[BOOTSTRAP_SERVERS],
    value_serializer=lambda x: json.dumps(x).encode('utf-8')
)

print("Cargando dataset...")
df = pd.read_csv(CSV_PATH)
print(f"Dataset cargado. Total de filas a simular: {len(df)}")

print("Enviando datos en streaming (Presiona Stop en Jupyter para parar)...")
try:
    for index, row in df.iterrows():
        transaction = row.to_dict()
        producer.send(TOPIC_NAME, value=transaction)
        print(transaction)
        # Enviamos 10 transacciones por segundo para probar
        time.sleep(0.1) 
        
        if index % 50 == 0 and index > 0:
            print(f">> Enviados {index} eventos a Kafka.")
except KeyboardInterrupt:
    print("\nSimulación detenida.")
finally:
    producer.flush()
    producer.close()

In [2]:
from os import listdir
CSV_PATH = '/work/data/creditcard.csv'
CSV_PATH = '/'
listdir(CSV_PATH)

['dev',
 'sys',
 'home',
 'etc',
 'lib64',
 'srv',
 'lib32',
 'usr',
 'mnt',
 'run',
 'tmp',
 'sbin',
 'bin',
 'libx32',
 'root',
 'media',
 'proc',
 'lib',
 'opt',
 'boot',
 'var',
 '.dockerenv']

In [3]:
from pyspark.sql import SparkSession
import time

# 1. Inicializar Spark en modo local (Ya lo tienes bien)
spark = SparkSession.builder \
    .appName("TestDeteccionFraude") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

# 2. Leer de Kafka con 'earliest' (Ya lo tienes bien)
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9093") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "earliest") \
    .load()

# 3. Pasar el payload binario a string legible (Ya lo tienes bien)
df_readable = df_kafka.selectExpr("CAST(value AS STRING) as json_payload")

# 4. CAMBIO CRÍTICO: En vez de 'console', guardamos en una tabla en MEMORIA llamada 'tabla_tfm'
query = df_readable.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("tabla_tfm") \
    .start()

# 5. Esperamos 10 segundos para dar tiempo a que Spark recoja los datos del pasado de Kafka
print("Conectando con Kafka y rellenando la tabla en memoria...")
time.sleep(10)

# 6. Forzamos a Jupyter a mostrar el contenido haciendo una consulta SQL a la memoria
print("\n--- ¡DATOS DETECTADOS EN TU PANTALLA! ---")
spark.sql("SELECT * FROM tabla_tfm LIMIT 10").show(truncate=False)

# 7. Detenemos el stream limpiamente
query.stop()
print("\nStream detenido con éxito.")

Conectando con Kafka y rellenando la tabla en memoria...


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: reentrant call inside <_io.BufferedReader name=51>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip


--- ¡DATOS DETECTADOS EN TU PANTALLA! ---


Py4JError: An error occurred while calling o47.showString

In [1]:
import pyspark
print(pyspark.__version__)

3.5.0


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, DoubleType
from pyspark.sql.functions import col, from_json
import time

# 1. Inicializar Spark con la versión exacta
spark = SparkSession.builder \
    .appName("TestDeteccionFraude") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

# 2. Definir el esquema exacto de tu CSV de Kaggle
schema = StructType([
    StructField("Time", DoubleType(), True),
    *[StructField(f"V{i}", DoubleType(), True) for i in range(1, 29)], # Genera V1 hasta V28
    StructField("Amount", DoubleType(), True),
    StructField("Class", DoubleType(), True) # Class es 0 (normal) o 1 (fraude)
])

# 3. Leer de Kafka
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9093") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "earliest") \
    .load()

# 4. Transformación Mágica: Convertir binario a String y luego desempaquetar el JSON
df_parsed = df_kafka \
    .selectExpr("CAST(value AS STRING) as json_string") \
    .select(from_json(col("json_string"), schema).alias("data")) \
    .select("data.*") # Esto extrae todas las variables en columnas separadas
print(df_parsed.columns)
# 5. Guardar en memoria para visualizar
query = df_parsed.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("tabla_tfm") \
    .start()

print("Conectando con Kafka, desempaquetando JSON y rellenando tabla...")
time.sleep(10)

print("\n--- ¡DATOS LISTOS PARA EL MODELO! ---")
spark.sql("SELECT Time, V1, V2, Amount, Class FROM tabla_tfm LIMIT 5").show(truncate=False)

query.stop()
print("\nStream detenido con éxito.")

['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
Conectando con Kafka, desempaquetando JSON y rellenando tabla...

--- ¡DATOS LISTOS PARA EL MODELO! ---
+--------+------------------+------------------+------+-----+
|Time    |V1                |V2                |Amount|Class|
+--------+------------------+------------------+------+-----+
|82959.0 |-0.277683866015678|1.01752679183182  |6.23  |NULL |
|14645.0 |1.20920884409233  |0.0144698003838745|24.15 |NULL |
|44322.0 |-0.481494520323824|0.801249264786701 |36.78 |NULL |
|165132.0|-7.50392623748137 |-0.360628009949399|12.31 |NULL |
|37616.0 |-1.05693880509791 |0.386879090160365 |0.64  |NULL |
+--------+------------------+------------------+------+-----+


Stream detenido con éxito.
